# 04 — External scenario datapackages

**Audience:** Users applying community or project-specific Frictionless datapackages through Premise.

**Prerequisites:** Network access or a local `datapackage.json`, a compatible ecoinvent database, and a valid IAM key for the accompanying IAM pathway.

**Learning goals:** inspect resources and declared scenarios, attach an external scenario to an IAM scenario, and apply the external transformation safely.


## Outline

1. Load and inspect a datapackage.
2. Select an exact external scenario label.
3. Attach it to a Premise scenario.
4. Apply external-only or combined updates.


In [ ]:
import os

import bw2data as bd
import pandas as pd
from datapackage import Package

from premise import NewDatabase

PACKAGE_URL = (
    "https://raw.githubusercontent.com/"
    "premise-community-scenarios/ammonia-prospective-scenarios/"
    "main/datapackage.json"
)
package = Package(PACKAGE_URL)


## 1. Inspect metadata and resources

Use labels declared by the package; capitalization and punctuation matter. Inspect resources before constructing `NewDatabase`.


In [ ]:
package_summary = {
    "name": package.descriptor["name"],
    "version": package.descriptor["version"],
    "ecoinvent": package.descriptor.get("ecoinvent"),
    "scenarios": package.descriptor.get("scenarios", []),
    "resources": package.resource_names,
}
package_summary


In [ ]:
scenario_data = pd.DataFrame(package.get_resource("scenario_data").read())
scenario_data.head()


## 2. Attach the external scenario

External scenarios belong inside each scenario dictionary under `external scenarios`. The older constructor-level `external_scenarios` argument should not be used.


In [ ]:
EXTERNAL_LABEL = "Business As Usual - image"
if EXTERNAL_LABEL not in package.descriptor["scenarios"]:
    raise ValueError(f"Unknown package scenario: {EXTERNAL_LABEL}")

external = {"scenario": EXTERNAL_LABEL, "data": package}
SCENARIOS = [
    {
        "model": "image",
        "pathway": "SSP2-M",
        "year": 2040,
        "external scenarios": [external],
    }
]


In [ ]:
PROJECT = "ecoinvent-3.10-cutoff"
SOURCE_DATABASE = "ecoinvent-3.10-cutoff"
BIOSPHERE_DATABASE = "ecoinvent-3.10-biosphere"
PREMISE_KEY = os.environ.get("PREMISE_KEY")

if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY before running this tutorial.")

bd.projects.set_current(PROJECT)
missing = [
    name for name in (SOURCE_DATABASE, BIOSPHERE_DATABASE) if name not in bd.databases
]
if missing:
    raise ValueError(f"Missing Brightway databases: {missing}")


## 3. Apply the package

Use `["external"]` to apply only the datapackage. Add IAM sectors when the research question also needs background transformations.


In [ ]:
ndb = NewDatabase(
    scenarios=SCENARIOS,
    source_db=SOURCE_DATABASE,
    source_version="3.10",
    biosphere_name=BIOSPHERE_DATABASE,
    key=PREMISE_KEY,
)

SECTORS = ["external"]
# Combined example: ["electricity", "fuels", "external"]
ndb.update(SECTORS)
ndb.write_db_to_brightway(name="premise-ammonia-bau-image-2040")


## Pitfalls and extension

- Confirm that the datapackage's ecoinvent metadata is compatible with the source database.
- Every configured resource path must resolve relative to `datapackage.json`.
- Scenario labels must match the package descriptor and scenario data exactly.
- Extension: replace `PACKAGE_URL` with a local `Path` and inspect the configuration and inventory resources before building.

## Exercise

Select the package's sustainable-development IMAGE scenario and prepare a combined electricity-plus-external update.


In [ ]:
exercise_external_label = "Sustainable development - image"
exercise_sectors = ["electricity", "external"]
